# Model Training and Evaluation

This notebook compares machine learning models using:
1. Custom URL features
2. Existing dataset features
3. Combined features

In [101]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [102]:
df = pd.read_csv('../data/processed/url_features.csv')
df.shape
df.head(10)

,normalized_url,url_length,hostname_length,path_length,query_length,path_depth,unique_query_parameters,num_digits,num_letters,num_dots,...,url_entropy,path_entropy,encoded_character_count,encoded_ratio,has_encoding,has_at_symbol,double_slash_in_path,fragment_length,has_fragment,label
0,http://nobell.it/70ffb52d079109dca5664cce6f317...,232.0,9.0,125.0,90.0,8.0,4.0,58.0,139.0,6.0,...,5.046608,4.760640,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,http://www.dghjdgf.com/paypal.co.uk/cycgi-bin/...,88.0,15.0,66.0,0.0,4.0,0.0,1.0,69.0,5.0,...,4.720685,4.630997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2,http://serviciosbys.com/paypal.cgi.bin.get-int...,184.0,16.0,161.0,0.0,10.0,0.0,47.0,115.0,7.0,...,4.739902,4.656329,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
3,http://mail.printakid.com/www.online.americane...,67.0,18.0,42.0,0.0,2.0,0.0,0.0,56.0,6.0,...,4.179853,4.014358,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,http://thewhiskeydregs.com/wp-content/themes/w...,123.0,19.0,60.0,36.0,6.0,0.0,21.0,86.0,1.0,...,4.623373,3.808719,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
5,http://smilesvoegol.servebbs.org/voegol.php,43.0,25.0,11.0,0.0,1.0,0.0,0.0,36.0,3.0,...,3.850716,3.095795,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
6,http://premierpaymentprocessing.com/includes/b...,68.0,28.0,33.0,0.0,2.0,0.0,7.0,51.0,2.0,...,4.524172,4.331977,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
7,http://myxxxcollection.com/v1/js/jih321/bpd.co...,67.0,19.0,41.0,0.0,6.0,0.0,4.0,49.0,5.0,...,4.326607,3.994764,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
8,http://super1000.info/docs,26.0,14.0,5.0,0.0,1.0,0.0,4.0,17.0,1.0,...,4.026987,2.321928,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
9,http://horizonsgallery.com/js/bin/ssl1/_id/www...,200.0,19.0,81.0,92.0,10.0,2.0,35.0,136.0,4.0,...,4.977870,4.460479,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [103]:
x = df.drop(columns=['normalized_url', 'label'])
y = df['label']

In [104]:
print("Shape of x:", x.shape)
print("Shape of y:", y.shape)

Shape of x: (95913, 40)
Shape of y: (95913,)


In [105]:
print("Label counts:")
print(y.value_counts())

print("\nLabel percentages:")
print((y.value_counts(normalize=True) * 100).round(2))

Label counts:
label
0    48009
1    47904
Name: count, dtype: int64

Label percentages:
label
0    50.05
1    49.95
Name: proportion, dtype: float64


In [106]:
constant_features = [
    col for col in x.columns
 if x[col].nunique() <= 1
 ]



In [107]:
print("Constant features:", constant_features)

Constant features: []


In [108]:
duplicate_count = x.duplicated().sum()
print("Duplicate feature rows:", duplicate_count)

Duplicate feature rows: 8547


#### GroupShuffleSplit ek hi group ke samples ko training aur testing dono mein jaane se rokta hai.

#### hum pehle duplicate feature combinations ko group karenge aur phir group-aware split karenge.

In [109]:
from sklearn.model_selection import GroupShuffleSplit


In [110]:
groups = x.groupby(
    list(x.columns),
    sort=False,
    dropna=False
).ngroup()

print("Total rows:", len(x))
print("Unique feature groups:", groups.nunique())
print("Duplicate feature rows:", groups.duplicated().sum())

Total rows: 95913
Unique feature groups: 87366
Duplicate feature rows: 8547


In [111]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(x, y, groups=groups)
)

x_train = x.iloc[train_idx]
x_test = x.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("Training features:", x_train.shape)
print("Testing features:", x_test.shape)

print("\nTraining labels:")
print(y_train.value_counts())

print("\nTesting labels:")
print(y_test.value_counts())

Training features: (77263, 40)
Testing features: (18650, 40)

Training labels:
label
0    38945
1    38318
Name: count, dtype: int64

Testing labels:
label
1    9586
0    9064
Name: count, dtype: int64


In [112]:
train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])

print("group overlapping:-" , len(train_groups & test_groups))

assert train_groups.isdisjoint(test_groups)
print("Verified: No identical feature group crosses the split.")


group overlapping:- 0
Verified: No identical feature group crosses the split.


In [113]:
from sklearn.linear_model import LogisticRegression

In [114]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000))
])

print("Logistic Regression model is ready.")

Logistic Regression model is ready.


In [115]:
model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('scaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](40,)","['url_length','hostname_length','path_length',...,'double_slash_in_path', 'fragment_length','has_fragment']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,40
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of

In [116]:
y_pred = model.predict(x_test)

In [117]:
print("actual labels" , y_test.head(15).tolist())
print("predicted labels" , y_pred[:15].tolist())

actual labels [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
predicted labels [1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [118]:
from sklearn.metrics import accuracy_score , f1_score , precision_score , recall_score , confusion_matrix

In [119]:
print("accuracy_score:" , np.round(accuracy_score(y_test, y_pred), 4))
print("Precision:", round(  precision_score(y_test, y_pred, zero_division=0), 4))
print("Recall:", round(recall_score(y_test, y_pred, zero_division=0), 4))
print("F1-score:", round(f1_score(y_test, y_pred, zero_division=0), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

accuracy_score: 0.8634
Precision: 0.916
Recall: 0.8085
F1-score: 0.8589

Confusion Matrix:
[[8353  711]
 [1836 7750]]


## decision tree model

In [120]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


In [121]:
dt_model = Pipeline([
    ("imputer" , SimpleImputer(strategy = "median")),
    ("classifier" , DecisionTreeClassifier(random_state = 42))
])

In [122]:
dt_model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](40,)","['url_length','hostname_length','path_length',...,'double_slash_in_path', 'fragment_length','has_fragment']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,40
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of 

In [123]:
dt_pred = dt_model.predict(x_test)


In [124]:
print("Actual labels:", y_test.head(10).tolist())
print("Predicted labels:", dt_pred[:10].tolist())

Actual labels: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
Predicted labels: [1, 0, 1, 1, 1, 1, 1, 1, 1, 1]


In [125]:
print("accuracy_score:" , np.round(accuracy_score(y_test, dt_pred), 4))
print("Precision:", round(  precision_score(y_test, dt_pred), 4))
print("Recall:", round(recall_score(y_test, dt_pred), 4))
print("F1-score:", round(f1_score(y_test, dt_pred), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, dt_pred))

accuracy_score: 0.8894
Precision: 0.8875
Recall: 0.8988
F1-score: 0.8931

Confusion Matrix:
[[7972 1092]
 [ 970 8616]]


# RANDOM FOREST

In [126]:
from sklearn.ensemble import RandomForestClassifier


In [127]:
rf_model = Pipeline([
    ("imputer",SimpleImputer(strategy = "median")),
    ("classifier" , RandomForestClassifier(n_estimators = 150 ,
                                           random_state = 42 , n_jobs = -1))
])


In [128]:
rf_model.fit(x_train, y_train)

print("Random Forest training completed!")

Random Forest training completed!


In [129]:
rf_pred = rf_model.predict(x_test)

In [130]:
print("accuracy_score:" , np.round(accuracy_score(y_test, rf_pred), 4))
print("Precision:", round(  precision_score(y_test, rf_pred), 4))
print("Recall:", round(recall_score(y_test, rf_pred), 4))
print("F1-score:", round(f1_score(y_test, rf_pred), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_pred))

accuracy_score: 0.9258
Precision: 0.945
Recall: 0.9085
F1-score: 0.9264

Confusion Matrix:
[[8557  507]
 [ 877 8709]]


# HistGradientBoostingClassifier

In [131]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [132]:
gb_model = Pipeline([
    ("imputer",SimpleImputer(strategy = "median")),
    ("classifier",HistGradientBoostingClassifier(random_state=42, max_iter=100))
])

In [133]:
gb_model.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](40,)","['url_length','hostname_length','path_length',...,'double_slash_in_path', 'fragment_length','has_fragment']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,40
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of 

In [134]:
gb_pred = gb_model.predict(x_test)

In [135]:
print("accuracy_score:" , np.round(accuracy_score(y_test, gb_pred), 4))
print("Precision:", round(  precision_score(y_test, gb_pred), 4))
print("Recall:", round(recall_score(y_test, gb_pred), 4))
print("F1-score:", round(f1_score(y_test, gb_pred), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, gb_pred))

accuracy_score: 0.9248
Precision: 0.9481
Recall: 0.9031
F1-score: 0.925

Confusion Matrix:
[[8590  474]
 [ 929 8657]]


## comparing results of 4 models

In [136]:
results = []

model_predictions = {
    "LogisticRegression":(model, y_pred),
    "DecisionTree":(dt_model, dt_pred),
    "RandomForest":(rf_model, rf_pred), 
    "HistGradientBoostingClassifier":(gb_model,gb_pred)
}
for name, (trained_model, predictions) in model_predictions.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test, predictions, zero_division=0
        ),
        "Recall": recall_score(
            y_test, predictions, zero_division=0
        ),
        "F1-score": f1_score(
            y_test, predictions, zero_division=0
        )
    })
results_df = pd.DataFrame(results)
display(
    results_df.sort_values(
        "F1-score", ascending=False
    ).round(4)
)

,Model,Accuracy,Precision,Recall,F1-score
2,RandomForest,0.9258,0.9450,0.9085,0.9264
3,HistGradientBoostingClassifier,0.9248,0.9481,0.9031,0.9250
1,DecisionTree,0.8894,0.8875,0.8988,0.8931
0,LogisticRegression,0.8634,0.9160,0.8085,0.8589


## Comparing Different Feature Sets

In [137]:
raw_df = pd.read_csv(
    "../data/raw/urlset.csv",
    encoding="latin1",
    on_bad_lines="skip"
)

print("Original dataset shape:", raw_df.shape)

print("\nColumn names:")
print(raw_df.columns.tolist())

Original dataset shape: (95913, 14)

Column names:
['domain', 'ranking', 'mld_res', 'mld.ps_res', 'card_rem', 'ratio_Rrem', 'ratio_Arem', 'jaccard_RR', 'jaccard_RA', 'jaccard_AR', 'jaccard_AA', 'jaccard_ARrd', 'jaccard_ARrem', 'label']


In [138]:
print("Target column present:", "label" in raw_df.columns)
print("URL column present:", "domain" in raw_df.columns)

print("\nLabel distribution:")
print(raw_df["label"].value_counts(dropna=False))

Target column present: True
URL column present: True

Label distribution:
label
0.0    48009
1.0    47904
Name: count, dtype: int64


In [139]:
existing_cols = [
    col for col in raw_df.columns
    if col not in ["domain", "label"]
]

print("Number of existing features:", len(existing_cols))
print("\nExisting feature names:")
print(existing_cols)

Number of existing features: 12

Existing feature names:
['ranking', 'mld_res', 'mld.ps_res', 'card_rem', 'ratio_Rrem', 'ratio_Arem', 'jaccard_RR', 'jaccard_RA', 'jaccard_AR', 'jaccard_AA', 'jaccard_ARrd', 'jaccard_ARrem']


In [143]:
custom_df = pd.read_csv("../data/processed/url_features.csv")

print("Custom dataset loaded!")
print("Shape:", custom_df.shape)

Custom dataset loaded!
Shape: (95913, 42)


In [144]:
x_existing = raw_df[existing_cols].copy()

print("Existing features shape:", x_existing.shape)
display(x_existing.head())

Existing features shape: (95913, 12)


,ranking,mld_res,mld.ps_res,card_rem,ratio_Rrem,ratio_Arem,jaccard_RR,jaccard_RA,jaccard_AR,jaccard_AA,jaccard_ARrd,jaccard_ARrem
0,10000000,1.0,0.0,18,107.611111,107.277778,0.0,0.0,0.0,0.0,0.8,0.795729
1,10000000,0.0,0.0,11,150.636364,152.272727,0.0,0.0,0.0,0.0,0.0,0.768577
2,10000000,0.0,0.0,14,73.500000,72.642857,0.0,0.0,0.0,0.0,0.0,0.726582
3,10000000,0.0,0.0,6,562.000000,590.666667,0.0,0.0,0.0,0.0,0.0,0.859640
4,10000000,0.0,0.0,8,29.000000,24.125000,0.0,0.0,0.0,0.0,0.0,0.748971


In [145]:
print("Original rows:", len(raw_df))
print("Custom feature rows:", len(custom_df))

assert len(raw_df) == len(custom_df), "Row counts do not match!"

assert raw_df["label"].astype(int).reset_index(drop=True).equals(
    custom_df["label"].astype(int).reset_index(drop=True)
), "Labels are not aligned!"

print("Row count and label alignment verified!")

Original rows: 95913
Custom feature rows: 95913
Row count and label alignment verified!


In [146]:
x_combined = pd.concat([ x_existing.reset_index(drop=True), x.reset_index(drop=True) ],
    axis=1
)

print("Existing features:", x_existing.shape)
print("Custom features:", x.shape)
print("Combined features:", x_combined.shape)

Existing features: (95913, 12)
Custom features: (95913, 40)
Combined features: (95913, 52)


In [147]:
raw_urls = raw_df["domain"].astype(str).apply(
    lambda url: url if url.startswith(("http://", "https://"))
    else "http://" + url
).reset_index(drop=True)

custom_urls = custom_df["normalized_url"].astype(str).reset_index(drop=True)

print("URLs match:", raw_urls.equals(custom_urls))

URLs match: True
